# Compression changes the Bayes denoising target

[Formal argument](../10_conditioning_denoising_projection.md). Full observation is noisy (90% accurate), not the hidden state. Both conditions use the same target, forward noise and schedule. Gaussian quadrature avoids confusing training noise with the analytic identity.

In [ ]:
import numpy as np

def denoising_witness(alpha=.8, sigma=.6, nodes_count=160):
    nodes, weights = np.polynomial.hermite.hermgauss(nodes_count)
    noise = np.sqrt(2.) * nodes
    weights = weights / np.sqrt(np.pi)
    result = {key: 0. for key in ['full_risk', 'compressed_risk', 'projection_gap',
                                 'cross_term', 'x0_gap', 'v_gap']}
    for z0 in (-1., 1.):
        for obs in (-1., 1.):
            mass = .5 * (.9 if obs == z0 else .1)
            zt = alpha * z0 + sigma * noise
            mean_full = np.tanh(alpha * zt / sigma**2 + .5 * obs * np.log(9.))
            mean_compressed = np.tanh(alpha * zt / sigma**2)
            eps_full = (zt - alpha * mean_full) / sigma
            eps_compressed = (zt - alpha * mean_compressed) / sigma
            v_full = alpha * eps_full - sigma * mean_full
            v_compressed = alpha * eps_compressed - sigma * mean_compressed
            result['full_risk'] += mass * (weights @ (noise - eps_full)**2)
            result['compressed_risk'] += mass * (weights @ (noise - eps_compressed)**2)
            result['projection_gap'] += mass * (weights @ (eps_full - eps_compressed)**2)
            result['cross_term'] += mass * (weights @ ((noise - eps_full) * (eps_full - eps_compressed)))
            result['x0_gap'] += mass * (weights @ (mean_full - mean_compressed)**2)
            result['v_gap'] += mass * (weights @ (v_full - v_compressed)**2)
    return {key: float(value) for key, value in result.items()}

result = denoising_witness()
finer = denoising_witness(nodes_count=240)
for key in result:
    np.testing.assert_allclose(result[key], finer[key], rtol=0, atol=2e-7)
assert result['full_risk'] > 0
assert result['projection_gap'] > 0
np.testing.assert_allclose(result['compressed_risk'] - result['full_risk'],
                           result['projection_gap'], rtol=0, atol=2e-7)
assert abs(result['cross_term']) < 1e-7
np.testing.assert_allclose(result['projection_gap'], (.8/.6)**2 * result['x0_gap'], atol=1e-12)
np.testing.assert_allclose(result['v_gap'], result['x0_gap']/.6**2, atol=1e-12)
print(result)
print('Quadrature refinement maximum change:', max(abs(result[k]-finer[k]) for k in result))

## Verify the score by differentiating the mixture density
This calculation is independent of substituting the score formula into itself.

In [ ]:
alpha, sigma = .8, .6
z = np.linspace(-3., 3., 301)
components = np.exp(-.5 * ((z[:, None] - alpha*np.array([-1., 1.]))/sigma)**2)
density = components.mean(axis=1)
derivative = (-(z[:, None] - alpha*np.array([-1., 1.]))/sigma**2 * components).mean(axis=1)
score_from_density = derivative/density
mean_z0 = np.tanh(alpha*z/sigma**2)
mean_eps = (z-alpha*mean_z0)/sigma
np.testing.assert_allclose(score_from_density, -mean_eps/sigma, atol=1e-12)
print('Conditional score maximum residual:', float(np.max(np.abs(score_from_density+mean_eps/sigma))))

## Boundary: one time point cannot certify sufficiency
At alpha=0, the noisy latent is pure noise and epsilon is known from it, even though the observation still contains information about Z0.

In [ ]:
boundary = denoising_witness(alpha=0., sigma=1.)
np.testing.assert_allclose(boundary['projection_gap'], 0., atol=1e-12)
assert boundary['x0_gap'] > .5
print('alpha=0:', boundary)
# Same weights on the same three times preserve the projection identity.
alphas = np.array([.2, .5, .8])
time_weights = np.array([.2, .3, .5])
checks = [denoising_witness(a, np.sqrt(1-a*a)) for a in alphas]
weighted_risk_gap = sum(w*(r['compressed_risk']-r['full_risk']) for w,r in zip(time_weights, checks))
weighted_projection = sum(w*r['projection_gap'] for w,r in zip(time_weights, checks))
np.testing.assert_allclose(weighted_risk_gap, weighted_projection, rtol=0, atol=2e-7)
print('Common-time-weight identity:', weighted_risk_gap, weighted_projection)

**Conclusion.** The two Bayes risks differ by the conditional-mean projection gap. This does not validate a finite neural architecture, prove sampler accuracy, or establish calibrated learned posteriors. The bounded-target corollary in the Markdown is not applied to an unbounded Gaussian coefficient prior.

In [ ]:
print("THEORY_DEMO_PASS::10_conditioning_denoising_projection")